<a href="https://colab.research.google.com/github/jaivaltamboli-pixel/SYNDROME/blob/main/GNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q stim pymatching torch_geometric scipy

In [ ]:
# Uncomment in a fresh Colab runtime:
# !pip install -q stim pymatching torch_geometric scipy matplotlib

import os
import gc
import json
import time
import random
from collections import deque
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import stim
import pymatching
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GINEConv, global_add_pool, global_mean_pool
from google.colab import drive

drive.mount("/content/drive")

SWEEP_DIR = "/content/drive/MyDrive/SYNDROME_QEC/SWEEP_SUBTHRESHOLD_v1"
os.makedirs(SWEEP_DIR, exist_ok=True)

SWEEP_GRID = [
    # (distance, total_depol_p, num_layers, train_val_shots)
    (3, 0.024, 4, 200_000),
    (3, 0.033, 4, 200_000),
    (3, 0.045, 4, 150_000),
    (5, 0.024, 6, 150_000),
    (5, 0.033, 6, 150_000),
    (5, 0.045, 6, 150_000),
]

HOLDOUT_SHOTS = 100_000
TRAIN_SEED = 42
HOLDOUT_SEED = 2026
HIDDEN_DIM = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"Device: {device} | BF16: {use_bf16} | Sweep directory: {SWEEP_DIR}")


def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def wilson_ci(errors: int, n: int, z: float = 1.96):
    p = errors / n
    denom = 1.0 + z**2 / n
    center = (p + z**2 / (2.0 * n)) / denom
    half = (z * np.sqrt((p * (1.0 - p) + z**2 / (4.0 * n)) / n)) / denom
    return max(0.0, center - half), min(1.0, center + half)


def build_circuit_and_topology(distance: int, total_p: float):
    p_single = total_p / 3.0
    base_circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_x",
        distance=distance,
        rounds=distance,
        after_clifford_depolarization=0.0,
        before_round_data_depolarization=1e-6,
        before_measure_flip_probability=p_single,
        after_reset_flip_probability=p_single,
    )
    circuit = stim.Circuit()
    for inst in base_circuit.flattened():
        if inst.name == "DEPOLARIZE1":
            targets = [t.value for t in inst.targets_copy()]
            circuit.append("PAULI_CHANNEL_1", targets, [p_single, p_single, p_single])
        else:
            circuit.append(inst)

    assert sum(1 for inst in circuit.flattened() if inst.name == "DEPOLARIZE1") == 0
    dem = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(dem)
    det_coords = circuit.get_detector_coordinates()

    num_dets = len(det_coords)
    coords_arr = np.array([det_coords[i][:3] for i in range(num_dets)], dtype=np.float32)
    coords_norm = coords_arr.copy()
    for axis in range(3):
        vals = np.sort(np.unique(coords_arr[:, axis]))
        step = float(np.min(np.diff(vals))) if len(vals) > 1 else 1.0
        coords_norm[:, axis] = (coords_arr[:, axis] - coords_arr[:, axis].mean()) / step

    bulk_edges, boundary_edges, edge_prob = {}, {}, {}
    adj = {i: set() for i in range(num_dets)}
    l0_seed_nodes = set()

    for u, v, data in matcher.edges():
        w = float(data["weight"])
        p = float(1.0 / (1.0 + np.exp(w)))
        crosses_log = 1.0 if 0 in data.get("fault_ids", set()) else 0.0
        u_i = int(u)
        if v is None or v == -1:
            boundary_edges[u_i] = (w, crosses_log)
            edge_prob[(u_i, -1)] = p
            if crosses_log > 0.5:
                l0_seed_nodes.add(u_i)
        else:
            a, b = min(u_i, int(v)), max(u_i, int(v))
            bulk_edges[(a, b)] = (w, crosses_log)
            edge_prob[(a, b)] = p
            adj[a].add(b)
            adj[b].add(a)
            if crosses_log > 0.5:
                l0_seed_nodes.update([a, b])

    x_comp_nodes = set(l0_seed_nodes)
    q = deque(l0_seed_nodes)
    while q:
        curr = q.popleft()
        for nxt in adj[curr]:
            if nxt not in x_comp_nodes:
                x_comp_nodes.add(nxt)
                q.append(nxt)
    z_comp_nodes = set(range(num_dets)) - x_comp_nodes

    x_list = sorted(x_comp_nodes)
    x_to_compact = {d: idx for idx, d in enumerate(x_list)}
    m_x = pymatching.Matching()
    x_edge_orig = {}

    for u, v, data in matcher.edges():
        w = float(data["weight"])
        fids = set(data.get("fault_ids", set()))
        u_i = int(u)
        if v is None or v == -1:
            if u_i in x_comp_nodes:
                x_edge_orig[(u_i, -1)] = (w, fids)
                m_x.add_boundary_edge(x_to_compact[u_i], weight=w, fault_ids=fids, merge_strategy="replace")
        else:
            a, b = min(u_i, int(v)), max(u_i, int(v))
            if a in x_comp_nodes and b in x_comp_nodes:
                x_edge_orig[(a, b)] = (w, fids)
                m_x.add_edge(x_to_compact[a], x_to_compact[b], weight=w, fault_ids=fids, merge_strategy="replace")

    z_to_x_joint, corr_probs = {}, {}
    for inst in dem.flattened():
        if inst.type != "error":
            continue
        p = float(inst.args_copy()[0])
        if p <= 0.0 or p >= 1.0:
            continue
        components = [[]]
        for target in inst.targets_copy():
            if target.is_separator():
                components.append([])
            elif target.is_relative_detector_id():
                components[-1].append(target.val)
        if len(components) < 2:
            continue

        x_keys, z_keys = [], []
        for comp in components:
            if len(comp) == 1:
                k = (comp[0], -1)
            elif len(comp) == 2:
                k = (min(comp[0], comp[1]), max(comp[0], comp[1]))
            else:
                continue
            (x_keys if comp[0] in x_comp_nodes else z_keys).append(k)

        for zk in z_keys:
            if zk in edge_prob:
                for xk in x_keys:
                    if xk in x_edge_orig:
                        d_map = z_to_x_joint.setdefault(zk, {})
                        prev = d_map.get(xk, 0.0)
                        d_map[xk] = prev * (1.0 - p) + p * (1.0 - prev)

        for i in range(len(components)):
            for j in range(i + 1, len(components)):
                for u in components[i]:
                    for v in components[j]:
                        if (u in x_comp_nodes) != (v in x_comp_nodes):
                            pair = (min(u, v), max(u, v))
                            prev = corr_probs.get(pair, 0.0)
                            corr_probs[pair] = prev * (1.0 - p) + p * (1.0 - prev)

    corr_edges = {}
    corr_adj = {i: set() for i in range(num_dets)}
    for (u, v), p in corr_probs.items():
        if 0.0 < p < 1.0:
            corr_edges[(u, v)] = float(np.log((1.0 - p) / p))
            corr_adj[u].add(v)
            corr_adj[v].add(u)

    topo = {
        "coords_norm": coords_norm, "bulk_edges": bulk_edges, "boundary_edges": boundary_edges,
        "adj": adj, "x_comp_nodes": x_comp_nodes, "z_comp_nodes": z_comp_nodes,
        "x_list": x_list, "x_to_compact": x_to_compact, "m_x": m_x,
        "edge_prob": edge_prob, "x_edge_orig": x_edge_orig,
        "z_to_x_joint": z_to_x_joint, "corr_edges": corr_edges, "corr_adj": corr_adj,
    }
    return circuit, matcher, topo


def run_corr_mwpm_and_build_graph(syn: np.ndarray, matcher: pymatching.Matching, topo: dict, actual_obs_bit: int = None):
    x_list = topo["x_list"]
    syn_x = syn[x_list]
    if not np.any(syn_x):
        return None, 0, False

    m_x, x_to_compact = topo["m_x"], topo["x_to_compact"]
    edge_prob, x_edge_orig = topo["edge_prob"], topo["x_edge_orig"]
    z_to_x_joint, z_comp_nodes = topo["z_to_x_joint"], topo["z_comp_nodes"]

    pass1_edges = matcher.decode_to_edges_array(syn)
    x_updates, chosen_z_bulk, chosen_z_b = {}, set(), set()

    for u, v in pass1_edges:
        u_i, v_i = int(u), int(v)
        if u_i not in z_comp_nodes:
            continue
        if v_i == -1:
            z_key = (u_i, -1)
            chosen_z_b.add(u_i)
        else:
            z_key = (min(u_i, v_i), max(u_i, v_i))
            chosen_z_bulk.add(z_key)
        corr_map = z_to_x_joint.get(z_key)
        if not corr_map:
            continue
        p_z_tot = edge_prob[z_key]
        for x_key, p_joint in corr_map.items():
            x_updates.setdefault(x_key, []).append((p_joint, min(p_joint / max(p_z_tot, 1e-12), 0.999999)))

    modified_keys, delta_w_map = [], {}
    try:
        for x_key, pairs in x_updates.items():
            orig_w, fids = x_edge_orig[x_key]
            p_curr = edge_prob[x_key]
            for p_joint, p_cond in pairs:
                p_uncorr = np.clip((p_curr - p_joint) / max(1.0 - 2.0 * p_joint, 1e-9), 1e-9, 1.0 - 1e-9)
                p_curr = p_uncorr * (1.0 - p_cond) + (1.0 - p_uncorr) * p_cond
            p_curr = float(np.clip(p_curr, 1e-9, 1.0 - 1e-9))
            w_new = float(np.log((1.0 - p_curr) / p_curr))
            delta_w_map[x_key] = orig_w - w_new
            a, b = x_key
            if b == -1:
                m_x.add_boundary_edge(x_to_compact[a], weight=w_new, fault_ids=fids, merge_strategy="replace")
            else:
                m_x.add_edge(x_to_compact[a], x_to_compact[b], weight=w_new, fault_ids=fids, merge_strategy="replace")
            modified_keys.append(x_key)
        pass2_compact = m_x.decode_to_edges_array(syn_x)
    finally:
        for x_key in modified_keys:
            orig_w, fids = x_edge_orig[x_key]
            a, b = x_key
            if b == -1:
                m_x.add_boundary_edge(x_to_compact[a], weight=orig_w, fault_ids=fids, merge_strategy="replace")
            else:
                m_x.add_edge(x_to_compact[a], x_to_compact[b], weight=orig_w, fault_ids=fids, merge_strategy="replace")

    bulk_edges, boundary_edges = topo["bulk_edges"], topo["boundary_edges"]
    corr_pred, touches_x_boundary = 0, False
    chosen_x_bulk, chosen_x_b = set(), set()

    for uc, vc in pass2_compact:
        u_g = x_list[int(uc)]
        if int(vc) == -1:
            touches_x_boundary = True
            chosen_x_b.add(u_g)
            if boundary_edges[u_g][1] > 0.5:
                corr_pred ^= 1
        else:
            v_g = x_list[int(vc)]
            pair = (min(u_g, v_g), max(u_g, v_g))
            chosen_x_bulk.add(pair)
            if bulk_edges[pair][1] > 0.5:
                corr_pred ^= 1

    if not touches_x_boundary:
        return None, corr_pred, False

    coords_norm, adj = topo["coords_norm"], topo["adj"]
    x_comp_nodes, corr_edges, corr_adj = topo["x_comp_nodes"], topo["corr_edges"], topo["corr_adj"]

    active_set = {int(d) for d in np.where(syn == 1)[0]}
    active_x = {d for d in active_set if d in x_comp_nodes}
    x_sub = set(active_x)
    for d in active_x:
        x_sub.update(adj[d])
    sub_nodes = set(x_sub)
    for d in x_sub:
        sub_nodes.update(corr_adj[d])
    sub_nodes = sorted(sub_nodes)

    det_to_local = {d: idx for idx, d in enumerate(sub_nodes)}
    node_features = [
        [coords_norm[d][0], coords_norm[d][1], coords_norm[d][2],
         1.0 if d in active_set else 0.0, 0.0,
         1.0 if d in boundary_edges else 0.0,
         1.0 if d in x_comp_nodes else 0.0]
        for d in sub_nodes
    ]

    chosen_bulk = chosen_x_bulk | chosen_z_bulk
    chosen_b = chosen_x_b | chosen_z_b
    edge_src, edge_dst, edge_attr = [], [], []

    def add_undirected(u_loc, v_loc, orig_w, delta_w, crosses_log, is_chosen, is_b, is_corr):
        feat = [orig_w, delta_w, crosses_log, is_chosen, is_b, is_corr]
        edge_src.extend([u_loc, v_loc])
        edge_dst.extend([v_loc, u_loc])
        edge_attr.extend([feat, feat])

    for idx_a, d_a in enumerate(sub_nodes):
        for d_b in adj[d_a]:
            if d_b > d_a and d_b in det_to_local:
                pair = (d_a, d_b)
                w, lf = bulk_edges[pair]
                add_undirected(idx_a, det_to_local[d_b], w, delta_w_map.get(pair, 0.0), lf, 1.0 if pair in chosen_bulk else 0.0, 0.0, 0.0)
        for d_b in corr_adj[d_a]:
            if d_b > d_a and d_b in det_to_local:
                pair = (d_a, d_b)
                add_undirected(idx_a, det_to_local[d_b], corr_edges[pair], 0.0, 0.0, 0.0, 0.0, 1.0)

    for d in sub_nodes:
        if d in boundary_edges:
            w, lf = boundary_edges[d]
            x, y, t = coords_norm[d]
            norm = float(np.hypot(x, y))
            ux, uy = (x / norm, y / norm) if norm > 1e-6 else (1.0, 0.0)
            v_loc = len(node_features)
            node_features.append([x + ux, y + uy, t, 0.0, 1.0, 1.0, 1.0 if d in x_comp_nodes else 0.0])
            add_undirected(det_to_local[d], v_loc, w, delta_w_map.get((d, -1), 0.0), lf, 1.0 if d in chosen_b else 0.0, 1.0, 0.0)

    if len(edge_src) == 0:
        return None, corr_pred, False

    y_tensor = torch.tensor([int(corr_pred != actual_obs_bit)], dtype=torch.float32) if actual_obs_bit is not None else None
    data = Data(
        x=torch.tensor(node_features, dtype=torch.float32),
        edge_index=torch.tensor([edge_src, edge_dst], dtype=torch.long),
        edge_attr=torch.tensor(edge_attr, dtype=torch.float32),
        y=y_tensor,
    )
    return data, corr_pred, True


class ResidualGINEBlock(nn.Module):
    def __init__(self, hidden_dim: int, edge_dim: int, dropout: float = 0.1):
        super().__init__()
        mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
        self.conv = GINEConv(nn=mlp, edge_dim=edge_dim, train_eps=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_attr):
        h = self.conv(x, edge_index, edge_attr)
        return self.norm(x + self.dropout(F.gelu(h)))


class SyndromeCorrArbiter(nn.Module):
    def __init__(self, node_dim: int = 7, edge_dim: int = 6, hidden_dim: int = 64, num_layers: int = 4, dropout: float = 0.1):
        super().__init__()
        self.node_enc = nn.Sequential(nn.Linear(node_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU())
        self.layers = nn.ModuleList([ResidualGINEBlock(hidden_dim, edge_dim, dropout) for _ in range(num_layers)])
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, data):
        x = self.node_enc(data.x)
        for layer in self.layers:
            x = layer(x, data.edge_index, data.edge_attr)
        g = torch.cat([global_add_pool(x, data.batch), global_mean_pool(x, data.batch)], dim=-1)
        return self.head(g).squeeze(-1)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cpu | BF16: False | Sweep directory: /content/drive/MyDrive/SYNDROME_QEC/SWEEP_SUBTHRESHOLD_v1


In [ ]:
def run_single_sweep_point(distance: int, total_p: float, num_layers: int, train_val_shots: int):
    tag = f"d{distance}_p{total_p:.3f}"
    res_path = os.path.join(SWEEP_DIR, f"{tag}_results.json")
    model_path = os.path.join(SWEEP_DIR, f"{tag}_model.pt")

    if os.path.exists(res_path):
        with open(res_path, "r") as f:
            r = json.load(f)
        print(f"[SKIP] {tag} already completed | Vanilla: {r['vanilla_ler']:.5f} | CorrMWPM: {r['corr_mwpm_ler']:.5f} | SYNDROME: {r['syndrome_ler']:.5f}")
        return r

    print(f"\n==================== RUNNING {tag} (layers={num_layers}, shots={train_val_shots:,}) ====================")
    set_all_seeds(TRAIN_SEED)
    circuit, matcher, topo = build_circuit_and_topology(distance, total_p)

    # 1. Generate train/val graphs
    t0 = time.perf_counter()
    sampler = circuit.compile_detector_sampler(seed=TRAIN_SEED)
    syndromes, actual_obs = sampler.sample(shots=train_val_shots, separate_observables=True)
    actual_bits = actual_obs[:, 0]

    all_graphs = []
    for i in range(train_val_shots):
        g, _, routed = run_corr_mwpm_and_build_graph(syndromes[i], matcher, topo, int(actual_bits[i]))
        if routed and g is not None:
            all_graphs.append(g)

    rng = random.Random(TRAIN_SEED)
    rng.shuffle(all_graphs)
    val_n = int(0.15 * len(all_graphs))
    val_graphs, train_graphs = all_graphs[:val_n], all_graphs[val_n:]
    print(f"Graphs ready in {time.perf_counter()-t0:.1f}s | Train: {len(train_graphs):,} | Val: {len(val_graphs):,} | Routing: {len(all_graphs)/train_val_shots:.1%}")

    # Pre-collate onto GPU
    train_batches = [Batch.from_data_list(train_graphs[i:i + 512]).to(device) for i in range(0, len(train_graphs), 512)]
    val_batches = [Batch.from_data_list(val_graphs[i:i + 1024]).to(device) for i in range(0, len(val_graphs), 1024)]
    del all_graphs, train_graphs, val_graphs, syndromes, actual_obs
    gc.collect()

    # 2. Train GINEConv arbiter
    set_all_seeds(TRAIN_SEED)
    model = SyndromeCorrArbiter(node_dim=7, edge_dim=6, hidden_dim=HIDDEN_DIM, num_layers=num_layers, dropout=0.1).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25)
    criterion = nn.BCEWithLogitsLoss()

    best_val_loss, best_epoch, no_imp = float("inf"), 0, 0
    for epoch in range(1, 26):
        model.train()
        random.shuffle(train_batches)
        tr_loss, tr_cnt = 0.0, 0
        for batch in train_batches:
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
                logits = model(batch)
                loss = criterion(logits, batch.y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()
            tr_loss += loss.item() * batch.num_graphs
            tr_cnt += batch.num_graphs

        model.eval()
        va_loss, va_cnt = 0.0, 0
        with torch.no_grad():
            for batch in val_batches:
                with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
                    va_loss += criterion(model(batch), batch.y).item() * batch.num_graphs
                va_cnt += batch.num_graphs
        scheduler.step()

        avg_va = va_loss / va_cnt
        if avg_va < best_val_loss - 1e-4:
            best_val_loss, best_epoch, no_imp = avg_va, epoch, 0
            torch.save(model.state_dict(), model_path)
        else:
            no_imp += 1
        if epoch % 5 == 0 or epoch == 1 or no_imp >= 6:
            print(f"  Epoch {epoch:02d} | Train BCE: {tr_loss/tr_cnt:.4f} | Val BCE: {avg_va:.4f}")
        if no_imp >= 6:
            break

    # 3. Validation threshold calibration
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()
    val_probs, val_targets = [], []
    with torch.no_grad():
        for batch in val_batches:
            with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
                val_probs.append(torch.sigmoid(model(batch)).float().cpu().numpy())
            val_targets.append(batch.y.cpu().numpy())
    val_probs, val_targets = np.concatenate(val_probs), np.concatenate(val_targets)

    best_thresh, best_gain = 0.50, -10**9
    for thresh in np.round(np.arange(0.45, 0.81, 0.05), 2):
        flips = val_probs >= thresh
        gain = int(np.sum(flips & (val_targets == 1))) - int(np.sum(flips & (val_targets == 0)))
        if gain > best_gain:
            best_gain, best_thresh = gain, float(thresh)
    if best_gain <= 0:
        best_thresh = 0.50

    del train_batches, val_batches
    torch.cuda.empty_cache()

    # 4. Holdout evaluation (100,000 fresh shots, seed=2026)
    h_sampler = circuit.compile_detector_sampler(seed=HOLDOUT_SEED)
    h_syn, h_obs = h_sampler.sample(shots=HOLDOUT_SHOTS, separate_observables=True)
    h_bits = h_obs[:, 0]

    vanilla_preds = matcher.decode_batch(h_syn)[:, 0]
    corr_preds = np.zeros(HOLDOUT_SHOTS, dtype=np.uint8)
    syn_preds = np.zeros(HOLDOUT_SHOTS, dtype=np.uint8)

    # Stream holdout inference in chunks of 2,048 graphs to keep RAM tiny
    buf_graphs, buf_idx = [], []
    routed_cnt = 0

    def flush_buffer():
        if not buf_graphs:
            return
        b = Batch.from_data_list(buf_graphs).to(device)
        with torch.no_grad():
            with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
                probs = torch.sigmoid(model(b)).float().cpu().numpy()
        for idx, prob in zip(buf_idx, probs):
            if prob >= best_thresh:
                syn_preds[idx] ^= 1
        buf_graphs.clear()
        buf_idx.clear()

    for i in range(HOLDOUT_SHOTS):
        g, c_pred, routed = run_corr_mwpm_and_build_graph(h_syn[i], matcher, topo)
        corr_preds[i] = c_pred
        syn_preds[i] = c_pred
        if routed and g is not None:
            routed_cnt += 1
            buf_graphs.append(g)
            buf_idx.append(i)
            if len(buf_graphs) >= 2048:
                flush_buffer()
    flush_buffer()

    van_err = vanilla_preds != h_bits
    cor_err = corr_preds != h_bits
    syn_err = syn_preds != h_bits

    van_ler, cor_ler, syn_ler = float(np.mean(van_err)), float(np.mean(cor_err)), float(np.mean(syn_err))
    tp_vs_cor = int(np.sum(cor_err & (~syn_err)))
    fp_vs_cor = int(np.sum((~cor_err) & syn_err))
    p_vs_cor = float(stats.binomtest(tp_vs_cor, tp_vs_cor + fp_vs_cor, p=0.5, alternative="two-sided").pvalue) if (tp_vs_cor + fp_vs_cor) > 0 else 1.0

    result = {
        "distance": distance,
        "total_p": total_p,
        "holdout_shots": HOLDOUT_SHOTS,
        "override_threshold": best_thresh,
        "val_max_prob": float(val_probs.max()),
        "vanilla_ler": van_ler,
        "vanilla_ci": list(wilson_ci(int(np.sum(van_err)), HOLDOUT_SHOTS)),
        "corr_mwpm_ler": cor_ler,
        "corr_mwpm_ci": list(wilson_ci(int(np.sum(cor_err)), HOLDOUT_SHOTS)),
        "syndrome_ler": syn_ler,
        "syndrome_ci": list(wilson_ci(int(np.sum(syn_err)), HOLDOUT_SHOTS)),
        "rel_vs_vanilla_pct": (van_ler - syn_ler) / max(van_ler, 1e-12) * 100.0,
        "rel_vs_corr_pct": (cor_ler - syn_ler) / max(cor_ler, 1e-12) * 100.0,
        "tp_vs_corr": tp_vs_cor,
        "fp_vs_corr": fp_vs_cor,
        "mcnemar_p_vs_corr": p_vs_cor,
        "routing_rate": routed_cnt / HOLDOUT_SHOTS,
    }
    with open(res_path, "w") as f:
        json.dump(result, f, indent=2)

    print(f"RESULT {tag} | Thresh={best_thresh:.2f} (val max_p={val_probs.max():.3f})")
    print(f"  Vanilla MWPM : {van_ler:.5f} | CorrMWPM : {cor_ler:.5f} | SYNDROME-on-Corr : {syn_ler:.5f}")
    print(f"  vs Vanilla   : {result['rel_vs_vanilla_pct']:+.2f}% | vs CorrMWPM : {result['rel_vs_corr_pct']:+.2f}% (TP={tp_vs_cor}, FP={fp_vs_cor}, p={p_vs_cor:.4g})")

    del model, h_syn, h_obs
    torch.cuda.empty_cache()
    gc.collect()
    return result


sweep_results = {}
for d, p, n_layers, n_shots in SWEEP_GRID:
    sweep_results[(d, p)] = run_single_sweep_point(d, p, n_layers, n_shots)


==================== RUNNING d3_p0.024 (layers=4, shots=200,000) ====================
Graphs ready in 71.0s | Train: 53,961 | Val: 9,522 | Routing: 31.7%
  Epoch 01 | Train BCE: 0.2981 | Val BCE: 0.2861
  Epoch 05 | Train BCE: 0.2493 | Val BCE: 0.2416
  Epoch 10 | Train BCE: 0.2150 | Val BCE: 0.2215
  Epoch 15 | Train BCE: 0.2087 | Val BCE: 0.2122
  Epoch 20 | Train BCE: 0.2036 | Val BCE: 0.2105
  Epoch 25 | Train BCE: 0.2022 | Val BCE: 0.2101
RESULT d3_p0.024 | Thresh=0.45 (val max_p=0.777)
  Vanilla MWPM : 0.03488 | CorrMWPM : 0.03351 | SYNDROME-on-Corr : 0.02970
  vs Vanilla   : +14.85% | vs CorrMWPM : +11.37% (TP=1181, FP=800, p=1.091e-17)

==================== RUNNING d3_p0.033 (layers=4, shots=200,000) ====================
Graphs ready in 69.9s | Train: 69,145 | Val: 12,202 | Routing: 40.7%
  Epoch 01 | Train BCE: 0.3582 | Val BCE: 0.3582
  Epoch 05 | Train BCE: 0.2960 | Val BCE: 0.3086
  Epoch 10 | Train BCE: 0.2716 | Val BCE: 0.2855
  Epoch 15 | Train BCE: 0.2646 | Val BCE: 0.

In [ ]:
p_vals = sorted({p for _, p in sweep_results.keys()})

print("================ SUB-THRESHOLD ERROR SUPPRESSION (Lambda = LER_d3 / LER_d5) ================")
print(f"{'p_total':>7} | {'Vanilla d3->d5 (Lambda)':>25} | {'CorrMWPM d3->d5 (Lambda)':>26} | {'SYNDROME d3->d5 (Lambda)':>26}")
print("-" * 95)
print("This is experimental value of Sweeping Edge Detection | Cell 3 values are going to be used for this test")

for p in p_vals:
    r3, r5 = sweep_results[(3, p)], sweep_results[(5, p)]
    lam_v = r3["vanilla_ler"] / max(r5["vanilla_ler"], 1e-12)
    lam_c = r3["corr_mwpm_ler"] / max(r5["corr_mwpm_ler"], 1e-12)
    lam_s = r3["syndrome_ler"] / max(r5["syndrome_ler"], 1e-12)
    print(
        f"{p:7.3f} | "
        f"{r3['vanilla_ler']:.5f}->{r5['vanilla_ler']:.5f} ({lam_v:4.2f}x) | "
        f"{r3['corr_mwpm_ler']:.5f}->{r5['corr_mwpm_ler']:.5f} ({lam_c:4.2f}x) | "
        f"{r3['syndrome_ler']:.5f}->{r5['syndrome_ler']:.5f} ({lam_s:4.2f}x)"
    )

# Publication-ready Log-Log Threshold Crossing Figure
fig, ax = plt.subplots(figsize=(8, 5.5), dpi=150)

styles = [
    ("vanilla_ler", "vanilla_ci", "Vanilla PyMatching", "#7f8c8d", ":"),
    ("corr_mwpm_ler", "corr_mwpm_ci", "Correlated MWPM (2-pass)", "#2980b9", "--"),
    ("syndrome_ler", "syndrome_ci", "SYNDROME-on-CorrMWPM", "#c0392b", "-"),
]

for key, ci_key, label, color, ls in styles:
    for d, marker in [(3, "o"), (5, "s")]:
        ys = [sweep_results[(d, p)][key] for p in p_vals]
        yerr_lo = [sweep_results[(d, p)][key] - sweep_results[(d, p)][ci_key][0] for p in p_vals]
        yerr_hi = [sweep_results[(d, p)][ci_key][1] - sweep_results[(d, p)][key] for p in p_vals]
        ax.errorbar(
            p_vals, ys, yerr=[yerr_lo, yerr_hi],
            label=f"{label} ($d={d}$)", color=color, linestyle=ls,
            marker=marker, markersize=6, linewidth=2.0 if d == 5 else 1.4,
            alpha=0.95 if d == 5 else 0.7, capsize=3,
        )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Physical Depolarizing Error Rate $p = p_X + p_Y + p_Z$", fontsize=11)
ax.set_ylabel("Logical Error Rate (LER)", fontsize=11)
ax.set_title("SYNDROME-on-CorrMWPM Sub-Threshold Scaling ($d=3$ vs. $d=5$)", fontsize=12, fontweight="bold")
ax.grid(True, which="both", linestyle="--", alpha=0.35)
ax.legend(fontsize=8.5, ncol=2, frameon=True)
plt.tight_layout()

plot_path = os.path.join(SWEEP_DIR, "threshold_scaling_d3_d5.png")
plt.savefig(plot_path, dpi=200)
plt.show()
print(f"Saved threshold figure to: {plot_path}")

In [ ]:
from torch_geometric.data import Batch

class ResidualGINEBlock(nn.Module):
    def __init__(self, hidden_dim: int, edge_dim: int, dropout: float = 0.1):
        super().__init__()
        mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.GELU(),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
        self.conv = GINEConv(nn=mlp, edge_dim=edge_dim, train_eps=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_attr):
        h = self.conv(x, edge_index, edge_attr)
        h = self.dropout(F.gelu(h))
        return self.norm(x + h)


class SyndromeCorrArbiter(nn.Module):
    def __init__(self, node_dim: int = 7, edge_dim: int = 6, hidden_dim: int = 96, num_layers: int = 6, dropout: float = 0.1):
        super().__init__()
        self.node_enc = nn.Sequential(
            nn.Linear(node_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.layers = nn.ModuleList([
            ResidualGINEBlock(hidden_dim, edge_dim, dropout=dropout) for _ in range(num_layers)
        ])
        # LayerNorm before head prevents global_add_pool magnitude explosion on large d=5 graphs
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, data):
        x = self.node_enc(data.x)
        for layer in self.layers:
            x = layer(x, data.edge_index, data.edge_attr)
        g = torch.cat([global_add_pool(x, data.batch), global_mean_pool(x, data.batch)], dim=-1)
        return self.head(g).squeeze(-1)


# Reload from memory or Drive cache if needed
if "train_graphs" not in globals() or "val_graphs" not in globals():
    train_graphs, val_graphs, dataset_stats = load_or_generate_datasets()
    INFER_GAP_CUTOFF = dataset_stats["infer_gap_cutoff"]

# Pre-collate once into GPU-resident batches (eliminates CPU collation bottleneck)
BATCH_SIZE = 512
rng = random.Random(CONFIG["train_seed"])
rng.shuffle(train_graphs)

print("Pre-collating train/val batches onto GPU...")
train_batches = [
    Batch.from_data_list(train_graphs[i:i + BATCH_SIZE]).to(device)
    for i in range(0, len(train_graphs), BATCH_SIZE)
]
val_batches = [
    Batch.from_data_list(val_graphs[i:i + 1024]).to(device)
    for i in range(0, len(val_graphs), 1024)
]

# Use BF16 automatically on L4/A100; stay FP32 on T4
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"Pre-collated {len(train_batches)} train batches | BF16 Tensor Cores: {use_bf16}")

set_all_seeds(CONFIG["train_seed"])
model = SyndromeCorrArbiter(
    node_dim=7,
    edge_dim=6,
    hidden_dim=CONFIG["hidden_dim"],
    num_layers=CONFIG["num_layers"],
    dropout=CONFIG["dropout"],
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
criterion = nn.BCEWithLogitsLoss()

t_train_start = time.perf_counter()
best_val_loss = float("inf")
best_epoch = 0
patience = 8
epochs_no_improve = 0

for epoch in range(1, 31):
    t_ep = time.perf_counter()
    model.train()
    train_loss_sum = 0.0
    random.shuffle(train_batches)

    for batch in train_batches:
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
            logits = model(batch)
            loss = criterion(logits, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()
        train_loss_sum += loss.item() * batch.num_graphs

    model.eval()
    val_loss_sum, val_total = 0.0, 0
    with torch.no_grad():
        for batch in val_batches:
            with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
                logits = model(batch)
                val_loss_sum += criterion(logits, batch.y).item() * batch.num_graphs
            val_total += batch.num_graphs

    scheduler.step()
    avg_train_loss = train_loss_sum / len(train_graphs)
    avg_val_loss = val_loss_sum / val_total

    if avg_val_loss < best_val_loss - 1e-4:
        best_val_loss = avg_val_loss
        best_epoch = epoch
        epochs_no_improve = 0
        torch.save(model.state_dict(), PATHS["model"])
    else:
        epochs_no_improve += 1

    print(f"Epoch {epoch:02d} ({time.perf_counter()-t_ep:.2f}s) | Train BCE: {avg_train_loss:.4f} | Val BCE: {avg_val_loss:.4f}")
    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch} (best epoch {best_epoch}).")
        break

print(f"\nTraining complete in {time.perf_counter() - t_train_start:.1f}s | Best Val BCE: {best_val_loss:.4f} (Epoch {best_epoch})")

# Calibration + tail diagnostic
model.load_state_dict(torch.load(PATHS["model"], map_location=device, weights_only=True))
model.eval()
val_probs, val_targets = [], []
with torch.no_grad():
    for batch in val_batches:
        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=use_bf16):
            val_probs.append(torch.sigmoid(model(batch)).float().cpu().numpy())
        val_targets.append(batch.y.cpu().numpy())

val_probs = np.concatenate(val_probs)
val_targets = np.concatenate(val_targets)

print(f"\nVal prob distribution: min={val_probs.min():.4f} | median={np.median(val_probs):.4f} | p99={np.quantile(val_probs, 0.99):.4f} | max={val_probs.max():.4f}")

calibration_records = []
best_thresh = 0.50
best_net_gain = -10**9

print("\n--- Validation Threshold Calibration (Overriding Correlated MWPM) ---")
for thresh in np.round(np.arange(0.45, 0.86, 0.05), 2):
    flips = val_probs >= thresh
    tp = int(np.sum(flips & (val_targets == 1)))
    fp = int(np.sum(flips & (val_targets == 0)))
    net_gain = tp - fp
    precision = tp / max(tp + fp, 1)
    calibration_records.append({
        "threshold": float(thresh), "flips": int(tp + fp),
        "tp_fixed": tp, "fp_broken": fp, "net_gain": net_gain, "precision": precision,
    })
    print(f"Thresh {thresh:.2f} | Flips: {tp+fp:4d} | TP: {tp:4d} | FP: {fp:4d} | Prec: {precision:.3f} | Net Fixed: {net_gain:+d}")
    if net_gain > best_net_gain:
        best_net_gain = net_gain
        best_thresh = float(thresh)

if best_net_gain <= 0:
    best_thresh = 0.50

with open(PATHS["calibration"], "w") as f:
    json.dump({
        "selected_threshold": best_thresh,
        "infer_gap_cutoff": INFER_GAP_CUTOFF,
        "best_val_net_gain": best_net_gain,
        "sweep": calibration_records,
    }, f, indent=2)
print(f"Selected Optimal Override Threshold: {best_thresh:.2f} (saved to Drive)")


In [ ]:
#4
def wilson_ci(errors: int, n: int, z: float = 1.96):
    p = errors / n
    denom = 1.0 + z**2 / n
    center = (p + z**2 / (2.0 * n)) / denom
    half = (z * np.sqrt((p * (1.0 - p) + z**2 / (4.0 * n)) / n)) / denom
    return max(0.0, center - half), min(1.0, center + half)


def evaluate_holdout():
    model.load_state_dict(torch.load(PATHS["model"], map_location=device, weights_only=True))
    model.eval()
    with open(PATHS["calibration"], "r") as f:
        cal = json.load(f)
    override_threshold = cal["selected_threshold"]
    infer_gap_cutoff = cal["infer_gap_cutoff"]

    holdout_shots = CONFIG["holdout_shots"]
    print(f"Sampling {holdout_shots:,} fresh d={CONFIG['distance']} holdout shots (seed={CONFIG['holdout_seed']})...")
    sampler = circuit.compile_detector_sampler(seed=CONFIG["holdout_seed"])
    syndromes, actual_obs = sampler.sample(shots=holdout_shots, separate_observables=True)
    actual_bits = actual_obs[:, 0]

    # 1. Vanilla PyMatching
    t0 = time.perf_counter()
    vanilla_preds = matcher.decode_batch(syndromes)[:, 0]
    t_van_us = (time.perf_counter() - t0) / holdout_shots * 1e6

    # 2. Exact Complementary Gap
    t0 = time.perf_counter()
    _, _, comp_gap = compute_exact_complementary_gap(m_gap, syndromes)
    t_gap_us = (time.perf_counter() - t0) / holdout_shots * 1e6

    # 3. Correlated MWPM + Full & Delta-W-Gated Graph Collection
    t0 = time.perf_counter()
    corr_preds = np.zeros(holdout_shots, dtype=np.uint8)
    full_graphs, full_indices = [], []
    gated_mask = comp_gap <= infer_gap_cutoff

    for i in range(holdout_shots):
        g, c_pred, routed = run_corr_mwpm_and_build_graph(
            syndromes[i], matcher, topo, build_graph=True
        )
        corr_preds[i] = c_pred
        if routed and g is not None:
            full_graphs.append(g)
            full_indices.append(i)
    t_corr_and_build_us = (time.perf_counter() - t0) / holdout_shots * 1e6

    # 4. Batched GNN inference
    t1 = time.perf_counter()
    full_syn_preds = corr_preds.copy()
    gated_syn_preds = corr_preds.copy()
    gated_routed_cnt = 0

    if full_graphs:
        eval_loader = DataLoader(full_graphs, batch_size=2048, shuffle=False, pin_memory=True)
        chunks = []
        with torch.no_grad():
            for batch in eval_loader:
                batch = batch.to(device, non_blocking=True)
                with torch.amp.autocast("cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                    chunks.append(torch.sigmoid(model(batch)).float().cpu().numpy())
        all_probs = np.concatenate(chunks)

        for idx, prob in zip(full_indices, all_probs):
            if prob >= override_threshold:
                full_syn_preds[idx] ^= 1
            if gated_mask[idx]:
                gated_routed_cnt += 1
                if prob >= override_threshold:
                    gated_syn_preds[idx] ^= 1
    t_gnn_us = (time.perf_counter() - t1) / holdout_shots * 1e6

    # Statistical Comparisons
    van_err = vanilla_preds != actual_bits
    cor_err = corr_preds != actual_bits
    van_ler, cor_ler = float(np.mean(van_err)), float(np.mean(cor_err))
    van_ci = wilson_ci(int(np.sum(van_err)), holdout_shots)
    cor_ci = wilson_ci(int(np.sum(cor_err)), holdout_shots)

    def summarize(name, syn_p, routed_cnt):
        s_err = syn_p != actual_bits
        s_ler = float(np.mean(s_err))
        s_ci = wilson_ci(int(np.sum(s_err)), holdout_shots)
        rel_vs_van = (van_ler - s_ler) / max(van_ler, 1e-12) * 100.0
        rel_vs_cor = (cor_ler - s_ler) / max(cor_ler, 1e-12) * 100.0
        tp = int(np.sum(cor_err & (~s_err)))
        fp = int(np.sum((~cor_err) & s_err))
        p_val = float(stats.binomtest(tp, tp + fp, p=0.5, alternative="two-sided").pvalue) if (tp + fp) > 0 else 1.0
        print(f"\n--- {name} ---")
        print(f"SYNDROME-on-CorrMWPM LER : {s_ler:.5f} (95% CI: [{s_ci[0]:.5f}, {s_ci[1]:.5f}])")
        print(f"vs. Vanilla PyMatching   : {rel_vs_van:+.2f}% relative reduction")
        print(f"vs. Correlated MWPM      : {rel_vs_cor:+.2f}% relative reduction | McNemar p = {p_val:.4g}")
        print(f"Overrides on CorrMWPM    : {tp} fixed / {fp} broken ({tp - fp:+d} net | Prec: {tp/max(tp+fp,1):.3f})")
        print(f"Routing Rate             : {routed_cnt/holdout_shots:.1%}")
        return {
            "ler": s_ler, "ci_95": list(s_ci), "rel_vs_vanilla_pct": rel_vs_van,
            "rel_vs_corr_pct": rel_vs_cor, "tp": tp, "fp": fp, "net": tp - fp, "mcnemar_p": p_val,
        }

    print(f"\n================ STACKED BENCHMARK ({RUN_ID}) ================")
    print(f"Holdout Shots: {holdout_shots:,} | Threshold: {override_threshold:.2f}")
    print(f"1. Vanilla PyMatching    : {van_ler:.5f} (95% CI: [{van_ci[0]:.5f}, {van_ci[1]:.5f}]) | {t_van_us:.2f} us/shot")
    print(f"2. Correlated MWPM       : {cor_ler:.5f} (95% CI: [{cor_ci[0]:.5f}, {cor_ci[1]:.5f}]) | vs Vanilla: {(van_ler-cor_ler)/van_ler*100:+.2f}%")

    res_full = summarize("3A. SYNDROME-on-CorrMWPM (Full X-Boundary)", full_syn_preds, len(full_graphs))
    res_gated = summarize(f"3B. SYNDROME-on-CorrMWPM (Delta-W Gated q{int(CONFIG['infer_gap_quantile']*100)})", gated_syn_preds, gated_routed_cnt)

    with open(PATHS["eval_results"], "w") as f:
        json.dump({
            "vanilla_ler": van_ler, "corr_mwpm_ler": cor_ler,
            "syndrome_full": res_full, "syndrome_gated": res_gated,
            "latency_us": {"vanilla": t_van_us, "comp_gap": t_gap_us, "corr_plus_build": t_corr_and_build_us, "gnn": t_gnn_us},
        }, f, indent=2)
    print(f"\nResults persisted to: {PATHS['eval_results']}")

evaluate_holdout()

Sampling 100,000 fresh d=5 holdout shots (seed=2026)...

================ STACKED BENCHMARK (d5_r5_onCorrMWPM_d1da8356) ================
Holdout Shots: 100,000 | Threshold: 0.45
1. Vanilla PyMatching    : 0.11179 (95% CI: [0.10985, 0.11376]) | 6.51 us/shot
2. Correlated MWPM       : 0.09424 (95% CI: [0.09244, 0.09607]) | vs Vanilla: +15.70%

--- 3A. SYNDROME-on-CorrMWPM (Full X-Boundary) ---
SYNDROME-on-CorrMWPM LER : 0.09302 (95% CI: [0.09124, 0.09484])
vs. Vanilla PyMatching   : +16.79% relative reduction
vs. Correlated MWPM      : +1.29% relative reduction | McNemar p = 0.002865
Overrides on CorrMWPM    : 885 fixed / 763 broken (+122 net | Prec: 0.537)
Routing Rate             : 84.0%

--- 3B. SYNDROME-on-CorrMWPM (Delta-W Gated q25) ---
SYNDROME-on-CorrMWPM LER : 0.09338 (95% CI: [0.09159, 0.09520])
vs. Vanilla PyMatching   : +16.47% relative reduction
vs. Correlated MWPM      : +0.91% relative reduction | McNemar p = 0.01946
Overrides on CorrMWPM    : 705 fixed / 619 broken (+86 n

In [ ]:
print(f"Val target base rate : {val_targets.mean():.4f} (Constant BCE baseline: {- (val_targets.mean()*np.log(val_targets.mean()) + (1-val_targets.mean())*np.log(1-val_targets.mean())):.4f})")
print(f"Val prob distribution: min={val_probs.min():.4f} | median={np.median(val_probs):.4f} | p99={np.quantile(val_probs, 0.99):.4f} | max={val_probs.max():.4f}")
for q in [0.95, 0.99, 0.995, 0.999]:
    cut = np.quantile(val_probs, q)
    mask = val_probs >= cut
    print(f"Top {(1-q)*100:4.1f}% (p >= {cut:.4f}) | n={mask.sum():4d} | Empirical CorrMWPM Error Rate: {val_targets[mask].mean():.3f}")

Val target base rate : 0.1096 (Constant BCE baseline: 0.3457)
Val prob distribution: min=0.0022 | median=0.0362 | p99=0.2732 | max=0.2749
Top  5.0% (p >= 0.2717) | n=2067 | Empirical CorrMWPM Error Rate: 0.369
Top  1.0% (p >= 0.2732) | n= 189 | Empirical CorrMWPM Error Rate: 0.270
Top  0.5% (p >= 0.2737) | n= 121 | Empirical CorrMWPM Error Rate: 0.289
Top  0.1% (p >= 0.2747) | n=  23 | Empirical CorrMWPM Error Rate: 0.304
